# Stage1 EXECUTE 目标的风险资产美元空位

按语义诊断只将实际提交的新目标用于投影空位核验；HOLD 仍留在全部会计窗口。

In [1]:
import hashlib,json,os
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_original_0bp_projection_dollar_slack_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['source'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
repair=json.loads((card/'execute_scope_repair_plan.json').read_text());assert plan['frozen'] and repair['frozen'] and (card/'run_state.json').exists() and not (card/'run_state_repair.json').exists()
sc=root/src['stage1_card'];ds=root/src['daily_card'];ss=json.loads((sc/'summary.json').read_text());dsummary=json.loads((ds/'summary.json').read_text())
assert sha(sc/'summary.json')==src['stage1_summary_sha256'] and sha(ds/'summary.json')==src['daily_summary_sha256']
assert ss['nas_evidence']['sha256']==src['stage1_NAS_sha256'] and dsummary['nas_evidence']['sha256']==src['daily_NAS_sha256'] and dsummary['nas_readback']['verified']
base=sc/'qlib_recorder'/src['stage1_experiment_id']/src['stage1_recorder_id']/'artifacts';assert sha(base/'inputs/frame')==src['stage1_frame_sha256'] and sha(base/'paths/A_active_research_replay_0')==src['stage1_path_sha256']
assert sha(ds/'qlib_recorder'/src['daily_experiment_id']/src['daily_recorder_id']/'artifacts/analysis/daily')==src['daily_artifact_sha256']
state={'status':'started','plan_sha256':sha(card/'plan.json'),'stage1_Recorder_loads':0,'daily_Recorder_loads':0,'new_model_fits':0,'new_native_accounts':0};(card/'run_state_repair.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1';qlib.init(provider_uri=str(root/src['provider_uri']),region='us')
with R.uri_context(str(sc/'qlib_recorder')):
    recorder=R.get_recorder(recorder_id=src['stage1_recorder_id'],experiment_id=src['stage1_experiment_id']);frame=recorder.load_object('inputs/frame').reset_index(drop=True);path=recorder.load_object('paths/A_active_research_replay_0')
state['stage1_Recorder_loads']=2;(card/'run_state_repair.json').write_text(json.dumps(state,indent=2)+'\n')
with R.uri_context(str(ds/'qlib_recorder')):
    daily=R.get_recorder(recorder_id=src['daily_recorder_id'],experiment_id=src['daily_experiment_id']).load_object('analysis/daily').reset_index(drop=True)
state['daily_Recorder_loads']=1;(card/'run_state_repair.json').write_text(json.dumps(state,indent=2)+'\n')
replay=path['replay'].reset_index(drop=True);targets=path['targets'].reset_index(drop=True)
assert len(frame)==len(replay)==len(targets)==len(daily)==944
for x in (replay,targets,daily):assert pd.to_datetime(frame.signal_date).equals(pd.to_datetime(x.signal_date))
cal=pd.DatetimeIndex(D.calendar(start_time='2023-01-03',end_time='2026-10-07',freq='day'));assert len(cal)==944 and pd.DatetimeIndex(pd.to_datetime(frame.signal_date)).equals(cal)
raw_soxl=frame.soxl_weight.to_numpy(float);raw_soxx=frame.soxx_weight.to_numpy(float);raw_risk=raw_soxl+raw_soxx;raw_beta=3*raw_soxl+raw_soxx
cap=replay.current_beta_cap.to_numpy(float);target_soxl=targets.soxl_weight.to_numpy(float);target_soxx=targets.soxx_weight.to_numpy(float);target_cash=targets.cash_weight.to_numpy(float);target_risk=target_soxl+target_soxx;target_beta=3*target_soxl+target_soxx
ceiling=np.minimum.reduce([raw_risk,cap,np.full(944,.85)]);slack=ceiling-target_risk
assert np.isfinite(np.c_[raw_risk,raw_beta,cap,target_risk,target_beta,target_cash]).all()
execute=replay.execute_open_target.to_numpy(bool)
assert np.max(np.abs(target_soxl+target_soxx+target_cash-1))<1e-9 and np.min(target_cash)>=.15-1e-9 and np.max(target_beta-cap)<1e-8 and np.min(slack[execute])>-1e-9
assert np.array_equal(execute,targets.execute_open_target.to_numpy(bool)) and np.array_equal(execute,daily.open_execute.to_numpy(bool))
binding=raw_beta>cap+1e-8;assert np.array_equal(binding,daily.cap_binding.to_numpy(bool))
eps=1e-9;strict_raw=(raw_risk+eps<cap)&(raw_risk+eps<.85);strict_cap=(cap+eps<raw_risk)&(cap+eps<.85);strict_floor=(.85+eps<raw_risk)&(.85+eps<cap)
category=np.where(strict_raw,'raw_risk',np.where(strict_cap,'beta_cap',np.where(strict_floor,'cash_floor','tie')))
assert np.isfinite(daily.cash_opportunity_log).all() and abs(float(daily.paired_log_excess.sum())-json.loads((ds/'result.json').read_text())['full']['relative_log_excess'])<1e-10
rows=[]
for kind in ('raw_risk','beta_cap','cash_floor','tie'):
    mask=execute&(category==kind);g=daily.loc[mask];rows.append({'constraint':kind,'EXECUTE_sessions':int(mask.sum()),'binding_sessions':int(np.sum(mask&binding)),'nonbinding_sessions':int(np.sum(mask&~binding)),'cash_opportunity_log':float(g.cash_opportunity_log.sum()),'paired_log_excess':float(g.paired_log_excess.sum()),'mean_target_cash':float(np.mean(target_cash[mask])) if mask.any() else None})
def periods(freq):
    result=[]
    groups=pd.to_datetime(daily.trade_date).dt.to_period(freq)
    for period,inds in pd.Series(range(944)).groupby(groups,sort=True):
        ids=inds.to_numpy();mask=execute[ids];c=category[ids]
        result.append({'period':str(period),'sessions':len(ids),'EXECUTE_sessions':int(mask.sum()),'EXECUTE_raw_risk_limited':int(np.sum(mask&(c=='raw_risk'))),'EXECUTE_beta_cap_limited':int(np.sum(mask&(c=='beta_cap'))),'EXECUTE_cash_floor_limited':int(np.sum(mask&(c=='cash_floor'))),'EXECUTE_ties':int(np.sum(mask&(c=='tie'))),'cash_opportunity_log':float(daily.cash_opportunity_log.iloc[ids].sum())})
    return result
window=126;rolling_excess=daily.paired_log_excess.rolling(window).sum().iloc[window-1:].reset_index(drop=True);assert len(rolling_excess)==819 and int((rolling_excess>0).sum())==255
window_cash={}
for kind in ('raw_risk','beta_cap','cash_floor','tie'):
    component=daily.cash_opportunity_log.where(execute&(category==kind),0.0).rolling(window).sum().iloc[window-1:].reset_index(drop=True)
    window_cash[kind]={'nonwinning_windows_with_negative_component':int(((rolling_excess<=0)&(component<0)).sum()),'mean_component_all_windows':float(component.mean())}
result={'decision':'read_only_projection_risky_dollar_ceiling_saturated_no_geometry_rescue' if np.max(np.abs(slack[execute]))<1e-8 else 'read_only_projection_material_EXECUTE_slack_found_no_policy','plan_sha256':state['plan_sha256'],'source_stage1_recorder_id':src['stage1_recorder_id'],'source_daily_recorder_id':src['daily_recorder_id'],'sessions':944,'EXECUTE_sessions':int(execute.sum()),'HOLD_sessions':int((~execute).sum()),'cap_binding_sessions':int(binding.sum()),'maximum_abs_EXECUTE_risky_dollar_slack':float(np.max(np.abs(slack[execute]))),'HOLD_carried_target_above_raw_ceiling_sessions':int(np.sum((~execute)&(slack<-1e-9))),'execute_scope_repair_plan_sha256':sha(card/'execute_scope_repair_plan.json'),'positive_slack_EXECUTE_sessions':int(np.sum(execute&(slack>1e-8))),'constraint_rows':rows,'annual':periods('Y'),'quarters':periods('Q'),'rolling126_windows':819,'original_positive_windows':255,'nonwinning_window_cash_components':window_cash,'theorem_boundary':'Under nonnegative SOXL/SOXX and beta=3*SOXL+SOXX, risky dollars<=beta cap; preserving raw risky-dollar intent and 15pct cash floor gives risky dollars<=min(raw risk, cap, .85). Saved Stage1 targets attain this bound. Changing raw intent or cap is a new allocator/risk policy.','new_market_requests':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Qlib_Recorders':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n');state['status']='completed';(card/'run_state_repair.json').write_text(json.dumps(state,indent=2)+'\n');print(json.dumps({k:result[k] for k in ['decision','EXECUTE_sessions','cap_binding_sessions','maximum_abs_EXECUTE_risky_dollar_slack','positive_slack_EXECUTE_sessions','constraint_rows']}))


{"decision": "read_only_projection_risky_dollar_ceiling_saturated_no_geometry_rescue", "EXECUTE_sessions": 797, "cap_binding_sessions": 288, "maximum_abs_EXECUTE_risky_dollar_slack": 1.1102230246251565e-16, "positive_slack_EXECUTE_sessions": 0, "constraint_rows": [{"constraint": "raw_risk", "EXECUTE_sessions": 600, "binding_sessions": 64, "nonbinding_sessions": 536, "cash_opportunity_log": -0.295864133733216, "paired_log_excess": -0.10863028194564148, "mean_target_cash": 0.4334635652899861}, {"constraint": "beta_cap", "EXECUTE_sessions": 197, "binding_sessions": 197, "nonbinding_sessions": 0, "cash_opportunity_log": -0.1563311980497054, "paired_log_excess": -0.15671204316279844, "mean_target_cash": 0.6510670065873282}, {"constraint": "cash_floor", "EXECUTE_sessions": 0, "binding_sessions": 0, "nonbinding_sessions": 0, "cash_opportunity_log": 0.0, "paired_log_excess": 0.0, "mean_target_cash": null}, {"constraint": "tie", "EXECUTE_sessions": 0, "binding_sessions": 0, "nonbinding_sessions

# 原 Stage1 0bp：合法投影已用满风险资产美元额

## 中文结论

复用冻结完整 Stage1 f696 分配器与 b472 行动、风控、恢复链的 0bp、扣 IBKR Tiered 费用原生 Qlib 账户。将全部 **944** 日的原分配输出、当日自有账户 beta 上限、保存目标和现金机会分量按信号日严格对齐；**797** 日实际提交开盘目标（EXECUTE），**147** 日 HOLD。两份来源 Recorder 的原件哈希与 Yanxuan NAS 冷回读资格均核对；0 新行情、拟合或账户。

对非负 SOXL/SOXX，风险资产美元权重 `r = SOXL + SOXX`，名义 beta `= r + 2×SOXL ≥ r`。若保持原分配提出的风险资产美元额为上界、当日 beta 上限以及 15% 最低现金，则 `r ≤ min(原风险资产额, beta 上限, 0.85)`。**797 个 EXECUTE 目标全部等于这个上界**，最大绝对误差约 **1.1e−16**；没有可通过只换合法投影几何填补的美元空位。实际执行目标的现金最少也高于 15%，所以 15% 现金底线没有成为紧约束。

在 EXECUTE 日，**600 日由原分配的风险资产意图限制**，现金机会分量合计 **−0.295864 对数点**；**197 日由 beta 上限限制**，合计 **−0.156331**。两者相加闭合为先前查得的 EXECUTE 日 **−0.452195**。原始名义 beta 超上限的 EXECUTE 日共 261 日，其中 **64 日仍由原风险资产额而非 beta 上限限制美元投入**；仅凭“名义 beta 受限”不能把全部现金都归因于风控。完整年度、季度以及 819 个连续 126 日窗口的分量计数见 `result.json`。

首次核查把保存的 `targets` 全部当作新投影目标，导致 18 条 HOLD 行的“原分配美元上界”断言失败。源回放在 HOLD 时保存的是**原持仓权重**，而非新提交的目标；这些行的权重闭合、现金底线和当日 beta 合法性均通过。失败与逐字段语义诊断保留在 `attempt1_failure.json`、`semantic_diagnostic_result.json`，一次机械修复仅把投影上界断言限定到 EXECUTE 日，未删 HOLD 日或改经济结果。

决定：`read_only_projection_risky_dollar_ceiling_saturated_no_geometry_rescue`。这只排除**同时保持原分配风险资产意图与原 beta 上限**的投影几何补救；提高原分配额度或放宽上限都是不同的自有状态策略，需要决策时可得且通过信息门的依据，并在完整 Stage1 派生 0bp Tiered 账户检验 SOXX 财富、回撤、CVaR、完整年、全部滚动窗和区块下界。本卡未运行候选账户，未证明超过 SOXX。
